
# 09 — Create Evaluation Test Set (Qwen-2.5-1.5B-Instruct on MPS)
**Project:** Semantic Book Recommender — 
IT4142 HUST
**Input:** `data/processed/books_with_emotions.csv`
**Output:**
- `data/eval/test_queries.json` — Evaluation dataset (100 queries)

Notebook này sinh tập câu hỏi đánh giá tự động bằng mô hình ngôn ngữ lớn chạy local.

## 1. Khởi tạo & Cấu hình đường dẫn

In [12]:
import pandas as pd
import json
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_PATH = Path('data/processed/books_with_emotions.csv')
EVAL_PATH = Path('data/eval/test_queries.json')
EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)

## 2. Load Dữ liệu Sách

In [13]:
print("Loading data...")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} books")

# Sample 100 books for evaluation
sample_df = df.sample(n=100, random_state=42).reset_index(drop=True)

Loading data...
Loaded 11,606 books


## 3. Khởi tạo Mô hình Qwen Local (MPS)

In [14]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print("Loading model and tokenizer...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Loaded model in {time.time()-t0:.1f}s on {model.device}")

Loading model and tokenizer...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded model in 23.7s on mps:0


## 4. Chuẩn bị Prompts

In [15]:
prompts = []
for _, row in sample_df.iterrows():
    desc = row['description']
    prompt_text = f"Describe a book with this plot: {desc}\nWrite a 1-sentence search query (no title, no author) to find this book. Return ONLY the search query."
    messages = [
        {"role": "system", "content": "You are a precise assistant. Write only the search query, nothing else."},
        {"role": "user", "content": prompt_text}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(formatted)
print(f"Prepared {len(prompts)} prompts")

Prepared 100 prompts


## 5. Sinh dữ liệu câu hỏi (Batched Generation) với Log Chi tiết

In [16]:
print("Starting generation...")
batch_size = 16
generated_queries = []
t_gen = time.time()
total_prompts = len(prompts)

for idx in range(0, total_prompts, batch_size):
    batch_idx = (idx // batch_size) + 1
    total_batches = (total_prompts + batch_size - 1) // batch_size
    
    batch_prompts = prompts[idx : idx + batch_size]
    batch_rows = sample_df.iloc[idx : idx + batch_size]
    
    print(f"\n--- [Batch {batch_idx}/{total_batches}] Processing items {idx} to {min(idx + batch_size, total_prompts)}... ---", flush=True)
    t_batch = time.time()
    
    inputs = tokenizer(batch_prompts, padding=True, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
        
    print(f"  Generated batch in {time.time() - t_batch:.2f}s. Decoding results...", flush=True)
    
    for i, (input_ids, out_ids) in enumerate(zip(inputs.input_ids, generated_ids)):
        gen_tokens = out_ids[len(input_ids):]
        query = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        query = query.replace('"', '').replace("'", '').strip()
        isbn = str(batch_rows.iloc[i]['isbn13'])
        title = str(batch_rows.iloc[i]['title'])
        
        print(f"    [{idx + i + 1}/100] ISBN: {isbn} | Title: '{title}'\n    => Query: \"{query}\"\n", flush=True)
        
        generated_queries.append({
            "query": query,
            "target_isbn13": isbn
        })
        
print(f"\n✓ All generation completed in {time.time() - t_gen:.1f}s", flush=True)

Starting generation...

--- [Batch 1/7] Processing items 0 to 16... ---
  Generated batch in 50.48s. Decoding results...
    [1/100] ISBN: 9781558322059 | Title: 'The Vegetarian Meat & Potatoes Cookbook'
    => Query: "Robin Robertson cookbook focusing on vegetarian meals satisfying both appetite and conscience"

    [2/100] ISBN: 9789181084498 | Title: 'The Music of Erich Zann'
    => Query: "Erich Zann mystery novel by H.P. Lovecraft"

    [3/100] ISBN: 9781250186935 | Title: 'Artificial Condition'
    => Query: "Artificial Condition by Ann Leckie"

    [4/100] ISBN: 9780429978715 | Title: 'Man the Hunted'
    => Query: "Man the Hunted: Evolutionary Perspectives on Human Origins and Prehistory"

    [5/100] ISBN: 9781936070657 | Title: 'Haiti Noir (Akashic Noir).'
    => Query: "Edwidge Danticat Haiti destitution literature"

    [6/100] ISBN: 9780763742904 | Title: 'Dietitian's Handbook of Enteral and Parenteral Nutrition'
    => Query: "Third Edition Handbook Systems Approaches Med

## 6. Lưu tập đánh giá

In [17]:
with open(EVAL_PATH, 'w', encoding='utf-8') as f:
    json.dump(generated_queries, f, indent=2)

print(f"✓ Saved {len(generated_queries)} queries to {EVAL_PATH}")

✓ Saved 100 queries to data/eval/test_queries.json
